# 05 — Train (config-driven, multi-GPU, auto fine-tune)

Trains the model defined in **`scripts/config.py`** in two phases, automatically:

1. **COARSE** — train up to `EPOCHS`, stopping early after `PATIENCE` epochs without
   validation improvement.
2. **FINE-TUNE** — take the **best** checkpoint from phase 1 and fine-tune it for
   `FINETUNE_EPOCHS` (default 75) more epochs at a lower learning rate.

Everything adjustable lives in `config.py`: **model size** (`n/s/m/l/x`), whether to
**build on an existing model** (`INIT_FROM`), the **classes**, the **augmentation**, the
schedule, and the GPUs (`GPUS="all"` uses every GPU of the profile). The batch size comes
from notebook 04.

> For a fully unattended, disconnect-proof run of **two** large models at once, use
> `python scripts/run_campaign.py` instead (see notebook `07_campaign`).

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/shared/s0598584/scripts')
import config as C
import train_pipeline as P

# One run named after the configured model size + image size.
RUN_NAME = f"{C.BASE_MODEL}{C.MODEL_SIZE}_{C.IMGSZ}_v1"
spec = P.make_spec(RUN_NAME, data=C.dataset_yaml(), imgsz=C.IMGSZ, model_size=C.MODEL_SIZE)

device, nproc = P.resolve_devices()
n_train = P._count_split(spec['data'], 'train')
n_val   = P._count_split(spec['data'], 'val')
cache   = P.auto_cache(spec, n_train, n_val, nproc)
total_batch, info = P.find_optimal_batch(spec, nproc)   # reuses notebook-04 result

print('='*60)
print('run        :', RUN_NAME)
print('init from  :', C.init_weights(C.MODEL_SIZE, C.INIT_FROM))
print('classes    :', C.class_names(), f'(nc={C.num_classes()})')
print('imgsz      :', C.IMGSZ, '| GPUs:', device, f'({nproc})')
print('batch      :', total_batch, f"(per-GPU {info['per_gpu_batch']} x {nproc})")
print('cache      :', cache, '| train/val:', n_train, '/', n_val)
print('coarse     :', C.EPOCHS, 'epochs, patience', C.PATIENCE)
print('fine-tune  :', C.FINETUNE_EPOCHS, 'epochs, lr0', C.FINETUNE_LR0)
print('='*60)

In [ ]:
# --- Phase 1: COARSE training until early stop ---
best = P.coarse_train(spec, device, total_batch, cache)
print('best (coarse):', best)

# --- Phase 2: FINE-TUNE the best checkpoint ---
final_best = P.finetune(spec, best, device, total_batch, cache)
print('\nFINAL model:', final_best)
print('Now run 06_evaluate.ipynb for test metrics + per-class best confidence.')